# Training base expert on vanilla OGBench environment using BC (cube)

In [ ]:
# import random
# import torch
# import os
# import math

# import matplotlib.pyplot as plt

# from collections import defaultdict

# from causal_gym import CubePCH
# from causal_rl.algo.imitation.imitate import *
# from causal_rl.algo.imitation.finetune import *

In [ ]:
# os.environ['CUDA_VISIBLE_DEVICES'] = '3'
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device

In [ ]:
# num_steps = 1000
# seed = 0
# hidden_dims = {'W'}

# random.seed(seed)
# torch.manual_seed(seed)

In [ ]:
# env = CubePCH(num_steps=num_steps, custom_hidden=hidden_dims, expert_mode=True, seed=seed, env_id='cube-quadruple-play-singletask-task2-v0')
# train_eps = env.expert.num_eps
# train_eps

In [ ]:
# X = {f'X{t}' for t in range(num_steps)}
# Y = f'Y{num_steps}'
# obs_prefix = env.env.observed_unobserved_vars[0]

In [ ]:
# Z_sets = {}
# for Xi in X:
#     i = int(Xi[1:])
#     cond = set()

#     for j in range(i+1):
#         cond.update({f'{o}{j}' for o in list(set(obs_prefix) - {'X'})})

#     for j in range(i):
#         cond.add(f'X{j}')

#     Z_sets[Xi] = cond

# Z_sets['X1']

In [ ]:
# records = collect_expert_trajectories(
#     env,
#     num_episodes=train_eps,
#     max_steps=num_steps,
#     seed=seed,
#     show_progress=True
# )

In [ ]:
# hidden_size = 256
# lr = 3e-4
# batch_size = 2048
# patience = 20
# lookback = 10
# num_blocks = 4
# epochs = 300
# dropout = 0.0

# dims = {
#     'Q': 6,
#     'V': 6,
#     'E': 3,
#     'H': 2,
#     'G': 1,
#     'C': 1,
#     'A': 3,
#     'L': 6,
#     'S': 3,
#     'M': 6,
#     'D': 3,
#     'N': 6,
#     'F': 3,
#     'O': 6,
#     'W': 2,
#     'X': 5
# }

In [ ]:
# model, slots, Z_trim = train_single_policy_long_horizon(
#     records,
#     Z_sets,
#     dims=dims,
#     epochs=epochs,
#     include_vars=obs_prefix,
#     lookback=lookback,
#     continuous=True,
#     num_actions = env.action_space.shape[0],
#     hidden_dim=hidden_size,
#     num_blocks=num_blocks,
#     dropout=dropout,
#     lr=lr,
#     batch_size=batch_size,
#     patience=patience,
#     device=device,
#     seed=seed,
#     action_bounds=(env.action_space.low, env.action_space.high)
# )

# policy = shared_policy_fn_long_horizon(model, slots, Z_trim, continuous=True, device=device)
# policies = make_shared_policy_dict(policy)

In [ ]:
# expert_episode_rewards = defaultdict(float)
# for rec in records:
#     ep = rec['episode']
#     expert_episode_rewards[ep] += float(rec['reward'])

# num_eps = len(expert_episode_rewards)
# expert_rewards = [expert_episode_rewards[e] for e in range(num_eps)]

# num_eval_eps = 20

# policy_records = collect_imitator_trajectories(
#     env=env,
#     policies=policies,
#     num_episodes=num_eval_eps,
#     max_steps=num_steps,
#     hidden_dims=hidden_dims,
#     show_progress=True
# )

# policy_episode_rewards = defaultdict(float)
# for rec in policy_records:
#     ep = rec['episode']
#     policy_episode_rewards[ep] += float(rec['reward'])

# policy_rewards = [policy_episode_rewards[e] for e in range(num_eval_eps)]

# sum(expert_rewards)/num_eps, sum(policy_rewards)/num_eval_eps

In [ ]:
# # save model for fine-tuning
# import os
# import torch

# SAVE_DIR = '/home/et2842/causal/causalrl/models'
# os.makedirs(SAVE_DIR, exist_ok=True)
# MODEL_PATH = os.path.join(SAVE_DIR, 'cube_quadruple_expert.pt')

# checkpoint = {
#     "state_dict": model.state_dict(),
#     "slots": slots,
#     "Z_trim": Z_trim,
#     "dims": dims,
#     "lookback": lookback,
#     "continuous": True,
#     "num_actions": env.action_space.shape[0],
#     "hidden_dim": hidden_size,
#     "num_blocks": num_blocks,
#     "dropout": 0.0,
#     "layernorm": True,
#     "final_tanh": True,
#     "action_bounds_low": env.action_space.low,
#     "action_bounds_high": env.action_space.high,
#     "input_dim": int(model.hidden.in_features),
# }

# torch.save(checkpoint, MODEL_PATH)
# print("Saved expert to:", MODEL_PATH)

# Fine-tuning expert on Cube Quadruple

In [ ]:
import random
import torch
import os
import math

import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import CubePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES'] = '3'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
# load model
MODEL_PATH = "/home/et2842/causal/causalrl/models/cube_quadruple_expert.pt"
checkpoint = torch.load(MODEL_PATH, map_location=device)

# Rebuild the model with the same architecture
action_bounds = (checkpoint['action_bounds_low'], checkpoint['action_bounds_high'])

pretrained_actor = ContinuousPolicyNN(
    input_dim=checkpoint['input_dim'],
    action_dim=checkpoint['num_actions'],
    hidden_dim=checkpoint['hidden_dim'],
    num_blocks=checkpoint['num_blocks'],
    dropout=checkpoint['dropout'],
    layernorm=checkpoint['layernorm'],
    final_tanh=checkpoint['final_tanh'],
    action_bounds=action_bounds,
).to(device)

pretrained_actor.load_state_dict(checkpoint['state_dict'])
pretrained_actor.train()

slots = checkpoint['slots']
Z_trim = checkpoint['Z_trim']
dims = checkpoint['dims']
lookback = checkpoint['lookback']

state_dim = checkpoint['input_dim']
state_dim, lookback

In [ ]:
num_steps = 1000
rl_seed_pretrain = 2014
rl_seed = 90210
hidden_dims = set() # {'W'}

env_pretrain = CubePCH(num_steps=num_steps, expert_mode=True, seed=rl_seed_pretrain, env_id='cube-quadruple-play-singletask-task2-v0')
env_train = CubePCH(num_steps=num_steps, expert_mode=True, seed=rl_seed, env_id='cube-quadruple-play-singletask-task2-v0')
action_dim = env_train.env.action_space.shape[0]
action_dim

In [ ]:
def make_dense_cube_reward(
    env,
    use_delta=True,
    c=5.0,
    per_cube_bonus=10.0,
    all_placed_bonus=50.0,
    success_threshold=0.04,
    time_penalty=0.01,
):
    """Dense shaped reward for cube-quadruple manipulation.

    Rewards per-cube progress toward goals using distance deltas,
    with bonuses for each cube placed and all cubes placed.
    Operates in scaled observation space for consistency.
    """
    XYZ_CENTER = np.array([0.425, 0.0, 0.0])
    XYZ_SCALER = 10.0
    pos_vars = ['A', 'S', 'D', 'F']
    scaled_threshold = success_threshold * XYZ_SCALER

    # Capture scaled goal positions from the environment
    base = env.env._get_base_env()
    scaled_goals = []
    for i in range(4):
        tar_pos = base._data.mocap_pos[base._cube_target_mocap_ids[i]].copy()
        scaled_goal = (tar_pos - XYZ_CENTER) * XYZ_SCALER
        scaled_goals.append(scaled_goal)

    def reward_fn(obs, reward_env):
        t = len(obs[pos_vars[0]]) - 1

        total_r = 0.0
        num_placed = 0

        for i, pv in enumerate(pos_vars):
            curr_pos = np.array(obs[pv][t], dtype=np.float64)
            dist_curr = np.linalg.norm(curr_pos - scaled_goals[i])

            # Delta shaping: reward progress toward goal
            if use_delta:
                if t > 0:
                    prev_pos = np.array(obs[pv][t - 1], dtype=np.float64)
                    dist_prev = np.linalg.norm(prev_pos - scaled_goals[i])
                    total_r += c * (dist_prev - dist_curr)
            else:
                total_r += -c * dist_curr

            # Per-cube success bonus
            if dist_curr <= scaled_threshold:
                total_r += per_cube_bonus
                num_placed += 1

        # All cubes placed bonus
        if num_placed == 4:
            total_r += all_placed_bonus

        # Time penalty
        total_r -= time_penalty

        return float(total_r)

    return reward_fn


reward_fn = make_dense_cube_reward(
    env_train,
    c=5.0,
    per_cube_bonus=10.0,
    all_placed_bonus=50.0,
    success_threshold=0.04,
    time_penalty=0.01,
)

In [ ]:
config = OnlineRLConfig(
    total_env_steps=1_000_000,
    start_steps=20_000,
    max_episode_steps=num_steps,
    batch_size=512,
    gamma=0.99,
    tau=0.005,
    policy_delay=2,
    actor_lr=3e-4,
    critic_lr=3e-4,
    noise_std=0.25,
    hidden_dim_q=512,
    target_policy_noise=0.2,
    target_noise_clip=0.3,
    actor_warmup_steps=100_000,
    bc_reg_lambda=0.01,
    max_grad_norm=1.0
)

In [ ]:
# pretrain critics offline
replay_buffer, q1, q2, target_q1, target_q2 = pretrain_critics_offline(
    env=env_pretrain,
    pretrained_actor=pretrained_actor,
    Z_trim=Z_trim,
    slots=slots,
    state_dim=state_dim,
    action_dim=action_dim,
    config=config,
    device=device,
    num_pretrain_steps=300_000,
    pretrain_updates=150_000,
    seed=rl_seed_pretrain,
    reward_shaping_fn=reward_fn
)

In [ ]:
def callback(stats: dict):
    if stats['episode'] % 1 == 0:
        print(
            f'[Episode {stats["episode"]}] '
            f'steps={stats["env_steps"]}, '
            f'return={stats["return"]:.2f}, '
            f'len={stats["length"]}, '
            f'buffer={stats["buffer_size"]}'
        )

In [ ]:
fine_tuned_policy, logs = td3_fine_tune_actor(
    env=env_train,
    actor=pretrained_actor,
    Z_trim=Z_trim,
    slots=slots,
    state_dim=state_dim,
    action_dim=action_dim,
    config=config,
    device=device,
    seed=rl_seed,
    log_callback=callback,
    replay_buffer=replay_buffer,
    initial_q1=q1,
    initial_q2=q2,
    initial_target_q1=target_q1,
    initial_target_q2=target_q2,
    reward_shaping_fn=reward_fn
)

ft_pi = shared_policy_fn_long_horizon(fine_tuned_policy, slots, Z_trim, continuous=True, device=device)
ft_policies = make_shared_policy_dict(ft_pi)

In [ ]:
expert_env = CubePCH(num_steps=num_steps, expert_mode=True, env_id='cube-quadruple-play-singletask-task2-v0')

In [ ]:
# save expert
import os
import torch

SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, 'cube_quadruple_expert_finetuned.pt')

checkpoint = {
    "state_dict": fine_tuned_policy.state_dict(),
    "slots": slots,
    "Z_trim": Z_trim,
    "dims": dims,
    "lookback": lookback,
    "continuous": True,
    "num_actions": env_train.action_space.shape[0],
    "hidden_dim": config.hidden_dim_q,
    "num_blocks": checkpoint['num_blocks'],
    "dropout": 0.0,
    "layernorm": True,
    "final_tanh": True,
    "action_bounds_low": env_train.action_space.low,
    "action_bounds_high": env_train.action_space.high,
    "input_dim": int(fine_tuned_policy.hidden.in_features),
}

torch.save(checkpoint, MODEL_PATH)
print("Saved expert to:", MODEL_PATH)

In [ ]:
num_eval_eps = 1000

expert_returns = collect_imitator_trajectories(
    env=expert_env,
    policies=ft_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True
)

In [ ]:
expert_episode_rewards = defaultdict(float)
for rec in expert_returns:
    ep = rec['episode']
    expert_episode_rewards[ep] += float(rec['reward'])

expert_rewards = [expert_episode_rewards[e] for e in range(num_eval_eps)]
sum(expert_rewards) / num_eval_eps

In [ ]:
mean_reward = np.mean(expert_rewards)
std_reward = np.std(expert_rewards)

print(f"E[Y]          = {mean_reward:.4f}")
print(f"Std[Y]        = {std_reward:.4f}")
print(f"E[Y] \u00b1 Std[Y] = {mean_reward:.4f} \u00b1 {std_reward:.4f}")

In [ ]:
# success rate: % of episodes where all cubes placed
ep_lengths = defaultdict(int)
for rec in expert_returns:
    ep_lengths[rec['episode']] += 1

lengths = np.array([ep_lengths[e] for e in range(num_eval_eps)])
successes = lengths < num_steps
success_rate = successes.mean()
se = np.sqrt(success_rate * (1 - success_rate) / num_eval_eps)

print(f"Success rate   = {100 * success_rate:.2f}% ({successes.sum()}/{num_eval_eps} episodes)")
print(f"Std error      = {100 * se:.2f}%")

In [ ]:
# successful episode lengths
success_lengths = lengths[successes]

if len(success_lengths) > 0:
    print(f"Successful episode lengths (n={len(success_lengths)}):")
    print(f"  Mean   = {np.mean(success_lengths):.2f}")
    print(f"  Std    = {np.std(success_lengths):.2f}")
    print(f"  Median = {np.median(success_lengths):.0f}")
    print(f"  Min    = {np.min(success_lengths)}")
    print(f"  Max    = {np.max(success_lengths)}")
    print(f"  25th%  = {np.percentile(success_lengths, 25):.0f}")
    print(f"  75th%  = {np.percentile(success_lengths, 75):.0f}")
else:
    print("No episodes were solved.")